In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


@dataclass(frozen=True)
class Paths:
    evaporation_workbook: Path = Path("data") / "evaporation_data.xlsx"
    pca_lda_workbook: Path = Path("data") / "data.xlsx"
    output_dir: Path = Path("outputs") / "evaporation"
    evaporation_train_sheet: str = "train"
    evaporation_test_sheet: str = "test"
    positive_sheet: str = "Positive"
    n_pca_components: int = 5


CLASS_ORDER = ["Lamp oil", "White spirit", "Diesel", "Gasoline"]
MASS_GROUP_DECIMALS = 4

LEVEL2_EXTERNAL_HOLDOUT_ROOTS = {
    "T1", "T2", "T4", "T5", "T7", "T10", "T11", "T13", "T14",
    "T6", "T9", "T12", "T15",
    "Te3", "Te6",
    "W1", "W2", "W3",
    "L1", "L3", "L7",
}

DIESEL_ROOTS = {
    "SH3", "SH4", "SH7", "SH8", "SH11", "SH12", "SH15", "SH16", "SH19", "SH20",
    "T3", "T6", "T9", "T12", "T15",
    "Te3", "Te6", "Te9", "Te12", "Te15",
}

GASOLINE_95_ROOTS = {
    "SH1", "SH5", "SH9", "SH13", "SH17",
    "T1", "T4", "T7", "T10", "T13",
    "Te1", "Te4", "Te7", "Te10", "Te13",
}

GASOLINE_98_ROOTS = {
    "SH2", "SH6", "SH10", "SH14", "SH18",
    "T2", "T5", "T8", "T11", "T14",
    "Te2", "Te5", "Te8", "Te11", "Te14",
}

GASOLINE_ROOTS = GASOLINE_95_ROOTS.union(GASOLINE_98_ROOTS)

MASS_COLUMN_CANDIDATES = [
    "mass (%)",
    "mass%",
    "mass",
    "remaining mass (%)",
    "remaining_mass_pct",
    "remaining mass",
    "mass of loss (%)",
    "mass loss (%)",
]


def root_code(sample_id: str) -> str:
    return str(sample_id).strip().split("-", 1)[0]


def normalize_column_name(value: str) -> str:
    return str(value).strip().lower()


def find_column(df: pd.DataFrame, candidates: List[str]) -> str | None:
    normalized_columns = {normalize_column_name(column): column for column in df.columns}

    for candidate in candidates:
        normalized_candidate = normalize_column_name(candidate)
        if normalized_candidate in normalized_columns:
            return normalized_columns[normalized_candidate]

    for column in df.columns:
        normalized_column = normalize_column_name(column)
        for candidate in candidates:
            if normalized_column.startswith(normalize_column_name(candidate)):
                return column
    return None


def spectral_columns_and_axis(df: pd.DataFrame) -> Tuple[List[str], np.ndarray]:
    columns: List[str] = []
    wavelengths: List[float] = []

    for column in df.columns:
        try:
            wavelength = float(str(column).strip())
        except (TypeError, ValueError):
            continue
        columns.append(column)
        wavelengths.append(wavelength)

    if not columns:
        raise ValueError("No numeric spectral columns were found.")

    wavelength_array = np.asarray(wavelengths, dtype=float)
    order = np.argsort(wavelength_array)
    return [columns[index] for index in order], wavelength_array[order]


def align_by_common_wavelengths(
    X_evaporation: np.ndarray,
    evaporation_wavelengths: np.ndarray,
    X_training: np.ndarray,
    training_wavelengths: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    evaporation_index = {
        float(wavelength): index
        for index, wavelength in enumerate(evaporation_wavelengths)
    }
    training_index = {
        float(wavelength): index
        for index, wavelength in enumerate(training_wavelengths)
    }
    common_wavelengths = sorted(
        set(evaporation_index).intersection(training_index)
    )
    if not common_wavelengths:
        raise ValueError(
            "No common wavelengths were found between the evaporation and PCA-LDA data."
        )

    evaporation_columns = [evaporation_index[value] for value in common_wavelengths]
    training_columns = [training_index[value] for value in common_wavelengths]
    return X_evaporation[:, evaporation_columns], X_training[:, training_columns]


def remaining_to_mass_loss_pct(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    finite_values = values[np.isfinite(values)]
    if finite_values.size == 0:
        return values.copy()
    if np.nanmax(np.abs(finite_values)) <= 1.5:
        return (1.0 - values) * 100.0
    return 100.0 - values


def format_mass_label(value: float) -> str:
    text = f"{float(value):.{MASS_GROUP_DECIMALS}f}".rstrip("0").rstrip(".")
    return text + "%"


def class_label_from_root(root: str) -> str:
    if root in DIESEL_ROOTS:
        return "Diesel"
    if root in GASOLINE_ROOTS:
        return "Gasoline"
    if root.startswith("L"):
        return "Lamp oil"
    if root.startswith("W"):
        return "White spirit"
    raise ValueError(f"Unknown root code for class mapping: {root}")


def load_pca_lda_development_data(
    paths: Paths,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    df = pd.read_excel(
        paths.pca_lda_workbook,
        sheet_name=paths.positive_sheet,
        index_col=0,
    )
    df.index = df.index.to_series().astype(str)
    df.columns = [str(column) for column in df.columns]

    roots = df.index.to_series().map(root_code).astype(str)
    keep_mask = ~roots.str.startswith("B")
    df = df.loc[keep_mask].copy()
    roots = roots.loc[keep_mask].copy()

    development_mask = ~roots.isin(LEVEL2_EXTERNAL_HOLDOUT_ROOTS)
    df = df.loc[development_mask].copy()
    roots = roots.loc[development_mask].copy()

    spectral_columns, wavelengths = spectral_columns_and_axis(df)
    X_raw = df[spectral_columns].to_numpy(dtype=float)
    labels = np.asarray(
        [class_label_from_root(root) for root in roots],
        dtype=object,
    )

    missing_classes = set(CLASS_ORDER).difference(np.unique(labels))
    if missing_classes:
        raise ValueError(
            f"PCA-LDA development data are missing classes: {sorted(missing_classes)}"
        )
    return X_raw, labels, wavelengths


def read_sheet_case_insensitive(path: Path, sheet_name: str) -> pd.DataFrame:
    workbook = pd.ExcelFile(path)
    matching_sheet = next(
        (
            sheet
            for sheet in workbook.sheet_names
            if str(sheet).strip().lower() == sheet_name.strip().lower()
        ),
        None,
    )
    if matching_sheet is None:
        raise ValueError(
            f"Sheet '{sheet_name}' was not found. Available sheets: {workbook.sheet_names}"
        )
    return pd.read_excel(path, sheet_name=matching_sheet, index_col=0)


def load_evaporation_data(
    paths: Paths,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    train = read_sheet_case_insensitive(
        paths.evaporation_workbook,
        paths.evaporation_train_sheet,
    )
    test = read_sheet_case_insensitive(
        paths.evaporation_workbook,
        paths.evaporation_test_sheet,
    )
    df = pd.concat([train, test], axis=0)
    df.columns = [str(column) for column in df.columns]

    mass_column = find_column(df, MASS_COLUMN_CANDIDATES)
    if mass_column is None:
        raise ValueError("No remaining-mass column was found.")

    remaining_mass = pd.to_numeric(df[mass_column], errors="coerce").to_numpy(dtype=float)
    if np.isnan(remaining_mass).any():
        raise ValueError("The remaining-mass column contains missing or non-numeric values.")

    mass_loss = remaining_to_mass_loss_pct(remaining_mass)
    mass_loss_group = np.round(mass_loss, decimals=MASS_GROUP_DECIMALS)

    spectral_columns, wavelengths = spectral_columns_and_axis(df)
    X_raw = df[spectral_columns].to_numpy(dtype=float)
    if np.isnan(X_raw).any():
        column_means = np.nanmean(X_raw, axis=0)
        missing_rows, missing_columns = np.where(np.isnan(X_raw))
        X_raw[missing_rows, missing_columns] = column_means[missing_columns]

    return mass_loss_group, X_raw, wavelengths


def preprocess_snv(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    means = X.mean(axis=1, keepdims=True)
    standard_deviations = X.std(axis=1, ddof=1, keepdims=True)
    standard_deviations[standard_deviations == 0.0] = 1.0
    return (X - means) / standard_deviations


def fit_pca_lda(
    X_development: np.ndarray,
    y_development: np.ndarray,
    n_pca_components: int,
) -> Tuple[PCA, LinearDiscriminantAnalysis]:
    components = min(
        n_pca_components,
        X_development.shape[1],
        X_development.shape[0] - 1,
    )
    if components < 2:
        raise ValueError("At least two PCA components are required.")

    pca = PCA(n_components=components)
    scores = pca.fit_transform(X_development)

    classes = np.unique(y_development)
    equal_priors = np.full(classes.shape[0], 1.0 / classes.shape[0])
    lda = LinearDiscriminantAnalysis(priors=equal_priors)
    lda.fit(scores, y_development)
    return pca, lda


def project_evaporation_spectra(
    pca: PCA,
    lda: LinearDiscriminantAnalysis,
    X_evaporation: np.ndarray,
) -> pd.DataFrame:
    scores = pca.transform(X_evaporation)
    log_probabilities = lda.predict_log_proba(scores)
    predicted_labels = lda.predict(scores)
    classes = np.asarray(lda.classes_, dtype=object)

    class_to_index = {class_name: index for index, class_name in enumerate(classes)}
    predicted_indices = np.asarray(
        [class_to_index[label] for label in predicted_labels],
        dtype=int,
    )
    row_indices = np.arange(log_probabilities.shape[0])
    second_indices = np.argsort(log_probabilities, axis=1)[:, -2]

    assigned_log_posterior = log_probabilities[row_indices, predicted_indices]
    competitor_log_posterior = log_probabilities[row_indices, second_indices]
    log10_lr = (
        assigned_log_posterior - competitor_log_posterior
    ) / np.log(10.0)

    return pd.DataFrame(
        {
            "predicted_label": predicted_labels,
            "log10_LR_assigned_vs_strongest_competitor": log10_lr,
        }
    )


def build_log_lr_table(
    mass_loss_group: np.ndarray,
    projections: pd.DataFrame,
) -> pd.DataFrame:
    per_spectrum = pd.DataFrame(
        {
            "Mass of loss (%)": mass_loss_group,
            "Mass label": [format_mass_label(value) for value in mass_loss_group],
        }
    )
    per_spectrum = pd.concat(
        [per_spectrum, projections.reset_index(drop=True)],
        axis=1,
    )

    rows = []
    group_columns = ["Mass of loss (%)", "Mass label", "predicted_label"]
    for (mass_value, mass_label, predicted_label), group in per_spectrum.groupby(
        group_columns,
        sort=True,
    ):
        log_lr_values = group[
            "log10_LR_assigned_vs_strongest_competitor"
        ].to_numpy(dtype=float)
        rows.append(
            {
                "Mass of loss (%)": mass_value,
                "Mass label": mass_label,
                "predicted_label": predicted_label,
                "n spectra": int(len(group)),
                "min log10 LR": float(np.nanmin(log_lr_values)),
                "max log10 LR": float(np.nanmax(log_lr_values)),
            }
        )

    return pd.DataFrame(
        rows,
        columns=[
            "Mass of loss (%)",
            "Mass label",
            "predicted_label",
            "n spectra",
            "min log10 LR",
            "max log10 LR",
        ],
    )


def run(paths: Paths | None = None) -> None:
    paths = Paths() if paths is None else paths
    paths.output_dir.mkdir(parents=True, exist_ok=True)

    mass_loss_group, X_evaporation_raw, evaporation_wavelengths = (
        load_evaporation_data(paths)
    )
    X_development_raw, y_development, development_wavelengths = (
        load_pca_lda_development_data(paths)
    )
    X_evaporation, X_development = align_by_common_wavelengths(
        X_evaporation_raw,
        evaporation_wavelengths,
        X_development_raw,
        development_wavelengths,
    )

    X_development_snv = preprocess_snv(X_development)
    X_evaporation_snv = preprocess_snv(X_evaporation)
    pca, lda = fit_pca_lda(
        X_development_snv,
        y_development,
        paths.n_pca_components,
    )
    projections = project_evaporation_spectra(
        pca,
        lda,
        X_evaporation_snv,
    )
    output_table = build_log_lr_table(mass_loss_group, projections)

    output_path = paths.output_dir / "evaporation_SNV_PCA_LDA_log_lr.xlsx"
    output_table.to_excel(output_path, index=False)
    print(f"Saved evaporation log(LR) table: {output_path}")


if __name__ == "__main__":
    run()
